Import libraries

In [1]:
env_config_path = r"..\exp_sets\wheeled\env1.ini" # Path for the env1 configuration
weights_path = rf"..\trained_models\wheeled\wheeled_env1_3robots_CrossQ.zip" # Path for trained model

In [2]:
import os
import datetime
import numpy as np
import configparser
import copy
import pygame
from shapely import Polygon, Point
import gymnasium as gym
from gymnasium import spaces

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


Necessary functions

In [3]:
def read_env_config(config_path):
    config = configparser.ConfigParser()
    config.read(config_path) # Read the config file
    env_params = {}

    # Load screen parameters
    env_params['SCREEN_WIDTH'] = float(config['SCREEN']['WIDTH'])
    env_params['SCREEN_HEIGHT'] = float(config['SCREEN']['HEIGHT'])

    # Load initial robot positions
    env_params['ROBOT_LENGTH'] = float(config['ROBOTS']['LENGTH'])
    env_params['ROBOT_WIDTH'] = float(config['ROBOTS']['WIDTH'])
    env_params['MAX_SPEED'] = float(config['ROBOTS']['MAX_SPEED'])
    env_params['MAX_STEER'] = float(config['ROBOTS']['MAX_STEER'])
    env_params['NUM_ROBOTS'] = int(config['ROBOTS']['NUM_ROBOTS'])
    env_params['ROBOT_INIT_CONFIGS'] = []
    for i in range(env_params['NUM_ROBOTS']):
        conf = config['ROBOTS'][f'ROBOT_{i+1}']
        x, y, theta = map(float, conf.split(','))
        # env_params[f'ROBOT_{i+1}'] = (x,y,theta)
        env_params['ROBOT_INIT_CONFIGS'].append((x,y,float(np.radians(theta))))

    # Load goal positions
    env_params['NUM_GOALS'] = int(config['GOALS']['NUM_GOALS'])
    env_params['GOAL_SIZE'] = float(config['GOALS']['GOAL_SIZE'])
    env_params['GOAL_POSITIONS'] = []
    for i in range(env_params['NUM_GOALS']):
        g_pos = config['GOALS'][f'GOAL_{i+1}']
        x, y = map(float, g_pos.split(','))
        # env_params[f'GOAL_{i+1}'] = (x,y)
        env_params['GOAL_POSITIONS'].append((x,y))

    # Load polygonal obstacles
    env_params['NUM_OBSTACLES'] = int(config['OBSTACLES']['NUM_OBSTACLES'])
    env_params['OBSTACLES'] = []
    for i in range(env_params['NUM_OBSTACLES']):
        ver_str = config['OBSTACLES'][f'OBSTACLE_{i+1}']
        points = [tuple(map(int, pt.split(','))) for pt in ver_str.split(';')]
        # env_params[f'OBSTACLE_{i+1}'] = (x,y)
        env_params['OBSTACLES'].append(points)    
    return env_params

def get_robot_polygon(x, y, theta, robot_length, robot_width):
    # Robot corners relative to center
    dx = robot_length / 2
    dy = robot_width / 2
    corners = np.array([
        [ dx,  dy],
        [ dx, -dy],
        [-dx, -dy],
        [-dx,  dy]
    ])

    # Rotation matrix
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    R = np.array([[cos_t, -sin_t], [sin_t, cos_t]])

    # Rotate and translate corners
    rotated = np.dot(corners, R.T) + np.array([x, y])
    return Polygon(rotated)

# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(int(b))
    # print('binary value:', bin)
    dec = int(bin,2)
    return dec


The multi-robot gymnasium environment:

In [4]:
class MultiRobotEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 60}
    def __init__(self, env_params, render_mode=None, scale=1.0):
        super().__init__()        
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        self.dt = 0.05  # timestep
        self.scale = scale

        # Screen parameters
        self.WIDTH  = int(env_params['SCREEN_WIDTH']  * self.scale)
        self.HEIGHT = int(env_params['SCREEN_HEIGHT'] * self.scale)

        # Robot parameters
        self.ROBOT_LENGTH = env_params['ROBOT_LENGTH'] * self.scale
        self.ROBOT_WIDTH  = env_params['ROBOT_WIDTH']  * self.scale
        self.MAX_SPEED    = env_params['MAX_SPEED'] * self.scale
        self.MAX_STEER    = np.radians(env_params['MAX_STEER'])  # angular, no scaling

        # Initial configurations
        self.NUM_ROBOTS = env_params['NUM_ROBOTS']
        self.init_ROBOTS = [
            (x * self.scale, y * self.scale, theta)
            for (x, y, theta) in env_params['ROBOT_INIT_CONFIGS']
        ]

        # Obstacles and goals
        self.goal_radius = env_params['GOAL_SIZE'] * self.scale
        self.goal_positions = [
            (gx * self.scale, gy * self.scale)
            for (gx, gy) in env_params['GOAL_POSITIONS']
        ]
        self.obstacles = [
            [(px * self.scale, py * self.scale) for (px, py) in obs]
            for obs in env_params['OBSTACLES']
        ]

        # Each robot's state: x, y, theta, v, delta
        obs_high = np.array(
            [self.WIDTH, self.HEIGHT, np.pi, self.MAX_SPEED, self.MAX_STEER] * self.NUM_ROBOTS,
            dtype=np.float64
        )
        obs_high = np.concatenate((obs_high, np.array([2**(len(self.obstacles))-1], dtype=np.float64)))

        obs_low = np.array(
            [0, 0, -np.pi, -self.MAX_SPEED, -self.MAX_STEER] * self.NUM_ROBOTS,
            dtype=np.float64
        )
        obs_low = np.concatenate((obs_low, np.array([0], dtype=np.float64)))
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float64)

        # Action: (accel, delta_change) for each robot
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.NUM_ROBOTS, 2), dtype=np.float64)

        # initialize reward values
        self.r_s, self.r_l, self.r_M = 10, 10000, 100000

        # define the sound file in the current directory
        self.sound_file_path = os.path.join(os.getcwd(),"misc", "boom.wav")
        self.play_sound = True if os.path.exists(self.sound_file_path) else False

        # Reset the environment and start
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.collision_occurred = False
        self.collision_point = (0,0)
        self.goal_visited = [False for _ in self.goal_positions]
        self.robot_paths = [[] for _ in range(self.NUM_ROBOTS)]

        # Initialize robot states
        v, delta = 0.0, 0.0 # Initial velocity and steering rate

        self.robots = [] # List to store the initial states of n robots
        self.configs = copy.deepcopy(self.init_ROBOTS) # Read the initial configurations

        for i in range(self.NUM_ROBOTS):
            x, y, theta = self.configs[i] # read config of each robot
            self.robots.append([x, y, theta, v, delta]) # state of each robot
        

        self.robots = np.array(self.robots, dtype=np.float64)
        self.t = 0

        if self.render_mode == "human":
            self._render_pygame()
        
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):        
        self.dec_g = binary_list_to_decimal(self.goal_visited) # Convert the visited goals to decimal value
        obs = np.concat((self.robots.flatten(), np.array([self.dec_g]))) # Add the encodings of goals
        info = {f'robot{i}': self.robots[i][:2] for i in range(self.NUM_ROBOTS)} # Current x,y positions of each robot
        return obs, info

    def step(self, action):
        terminated, truncated = False, False        
        reward = - (self.r_s / self.dec_g) if (self.dec_g != 0) else -self.r_s  # base step penalty
        self.t += 1 # step_count
        robot_polygons = [] # to store the space occupied by each robot

        for i in range(self.NUM_ROBOTS): # For every robot
            a = float(action[i][0]) * 100.0 # Acceleration 
            d_delta = float(action[i][1]) #* self.ds # Rate of change in steering
            x, y, theta, v, delta = self.robots[i] # get the current state

            # Update state
            v = np.clip(v + a * self.dt, -self.MAX_SPEED, self.MAX_SPEED)
            delta = np.clip(delta + d_delta, -self.MAX_STEER, self.MAX_STEER)

            if abs(delta) > 1e-4: # avoid division by zero or numerical instability
                R = self.ROBOT_LENGTH / np.tan(delta)
                omega = v / R
            else:
                omega = 0.0 # go straight for a very small delta

            theta += omega * self.dt
            x += v * np.cos(theta) * self.dt
            y += v * np.sin(theta) * self.dt

            # Keep inside bounds
            x = np.clip(x, 0, self.WIDTH)
            y = np.clip(y, 0, self.HEIGHT)            

            # Make the robot polygon object
            robot_poly = get_robot_polygon(x, y, theta, self.ROBOT_LENGTH, self.ROBOT_WIDTH)

            # Check for collisions with obstacles                       
            for obs_pts in self.obstacles: 
                obs_poly = Polygon(obs_pts)
                if robot_poly.intersects(obs_poly): # if collisions occur
                    inter = robot_poly.intersection(obs_poly)
                    int_point = inter.representative_point()
                    self.collision_point = (int_point.x, int_point.y)
                    reward = -self.r_M # very large negative reward
                    terminated = True
                    self.collision_occurred = True
            
            # Check for collisions between robots
            if robot_polygons: # If there exists other robot polygons
                for robot_i in robot_polygons: # For each robot polygon
                    if robot_poly.intersects(robot_i): # if collisions occur
                        inter = robot_poly.intersection(robot_i)
                        int_point = inter.representative_point()
                        self.collision_point = (int_point.x, int_point.y)
                        reward = -self.r_M
                        terminated = True
                        self.collision_occurred = True                      
            robot_polygons.append(robot_poly) # Add the current robot to the robot polygons

            # Check if a goal region is visited
            for j, (gx, gy) in enumerate(self.goal_positions): # Loop through each goal region
                if not self.goal_visited[j]:
                    goal_point = Point(gx,gy)
                    goal_point = goal_point.buffer(self.goal_radius) # define the goal region
                    if goal_point.intersects(robot_poly): # If the goal point is reached                        
                        reward += self.r_l # Get a large reward for visiting each goal region                        
                        self.goal_visited[j] = True
                        print(f"GOAL {j} reached!!! Goals visited: {sum(self.goal_visited)}, reward: {reward}")
            
            # Record the robot path
            self.robots[i] = [x, y, theta, v, delta]
            self.robot_paths[i].append((x, y))

        # Check if all goal regions are visited
        if sum(self.goal_visited) >= len(self.goal_positions):
            reward += self.r_M
            print(f"All goals reached!!! Goals visited: {self.goal_visited}, reward: {reward}")
            terminated = True

        if self.render_mode == "human":
            self._render_pygame()
        
        obs, info = self._get_obs() # Get the updated observations
        return obs, reward, terminated, truncated, info
    
    def _render_pygame(self):
        if self.screen is None:
            pygame.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-Robot Environment")
            self.clock = pygame.time.Clock()

        self.screen.fill((240, 240, 240))

        # Draw obstacles
        for obs in self.obstacles:
            pygame.draw.polygon(self.screen, (100, 100, 100), obs)

        # Draw goal regions    
        for (gx, gy), visited in zip(self.goal_positions, self.goal_visited):
            color = (0, 255, 100) if not visited else (0, 0, 0)
            pygame.draw.circle(
                self.screen,
                color,
                (int(gx), int(gy)),
                int(self.goal_radius),  # already scaled in __init__
                0
            )

        # Draw collision point if collision occurred
        if self.collision_occurred:
            pygame.draw.circle(
                self.screen,
                color=(255, 0, 0),
                center=(int(self.collision_point[0]), int(self.collision_point[1])),
                radius=int(10 * self.scale)  # scaled explosion radius
            )
            if self.play_sound:            
                exp_sound = pygame.mixer.Sound(self.sound_file_path)
                exp_sound.play()

        # Draw robot paths
        for path in self.robot_paths:
            if len(path) > 1:
                pygame.draw.lines(
                    self.screen,
                    color=[255, 0, 0],
                    closed=False,
                    points=path,
                    width=int(2 * self.scale)  # scaled path thickness
                )

        # Draw each robot
        for x, y, theta, _, delta in self.robots:
            # --- Draw robot body ---
            robot_surf = pygame.Surface((self.ROBOT_LENGTH, self.ROBOT_WIDTH), pygame.SRCALPHA)
            robot_surf.fill((0, 128, 255))
            pygame.draw.line(
                robot_surf,
                (255, 0, 0),
                (self.ROBOT_LENGTH // 2, self.ROBOT_WIDTH // 2),
                (self.ROBOT_LENGTH, self.ROBOT_WIDTH // 2),
                int(2 * self.scale)  # scaled front arrow
            )
            rotated_robot = pygame.transform.rotate(robot_surf, -np.degrees(theta))
            rect = rotated_robot.get_rect(center=(x, y))
            self.screen.blit(rotated_robot, rect)

            # --- Draw wheels ---
            offset_x = self.ROBOT_LENGTH // 2 - self.ROBOT_WIDTH / 5
            offset_y = self.ROBOT_WIDTH // 2 - self.ROBOT_WIDTH / 5
            wheel_offsets = {
                'front_left':  (offset_x, -offset_y),
                'front_right': (offset_x, offset_y),
                'rear_left':   (-offset_x, -offset_y),
                'rear_right':  (-offset_x, offset_y),
            }

            for pos, (dx, dy) in wheel_offsets.items():
                # Rotate wheel positions into global frame
                x_local = dx
                y_local = dy
                x_global = x + np.cos(theta) * x_local - np.sin(theta) * y_local
                y_global = y + np.sin(theta) * x_local + np.cos(theta) * y_local

                # Wheel surface
                wheel_surf = pygame.Surface(
                    (self.ROBOT_LENGTH / 6, max(1, int(1 * self.scale))),
                    pygame.SRCALPHA
                )
                wheel_surf.fill((20, 20, 20))

                # Determine wheel rotation
                if 'front' in pos:
                    wheel_angle = theta + delta
                else:
                    wheel_angle = theta

                rotated_wheel = pygame.transform.rotate(wheel_surf, -np.degrees(wheel_angle))
                wheel_rect = rotated_wheel.get_rect(center=(x_global, y_global))
                self.screen.blit(rotated_wheel, wheel_rect)

        pygame.display.flip()
        self.clock.tick(60)
        pygame.event.get()

    def render(self):
        self._render_pygame()

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None


Register the environment

In [5]:
# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=1000)

Trained Inference:

In [6]:
from sb3_contrib import CrossQ
envid = "env1"

# Load trained network
model = CrossQ.load(weights_path)

In [7]:
env_params = read_env_config(env_config_path)

env = MultiRobotEnv(env_params=env_params, render_mode="human", scale=1.0)
obs, info = env.reset()
print("observation:", obs)
env.render()
assert False, "Manual break for positioning Pygame window"

observation: [ 3.50000000e+02  7.50000000e+01  7.85398163e-01  0.00000000e+00
  0.00000000e+00  1.00000000e+02  1.50000000e+02 -2.61799388e-01
  0.00000000e+00  0.00000000e+00  2.00000000e+02  2.25000000e+02
 -3.12413936e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]


AssertionError: Manual break for positioning Pygame window

In [8]:
total_rewards = 0
total_steps = 0

for i in range(10000):
    action, _ = model.predict(obs) # model prediction
    obs, reward, terminated, truncated, info = env.step(action)
    total_rewards += reward
    total_steps += 1
    if terminated or truncated:
        print(f"Episode ended. total timesteps:{total_steps}, total rewards: {total_rewards}")
        print(f"Current observation: {obs}")
        pygame.time.wait(2000)
        # obs, _ = env.reset()
        # total_rewards, total_steps = 0, 0
        break
    pygame.event.get()

GOAL 0 reached!!! Goals visited: 1, reward: 9990
GOAL 2 reached!!! Goals visited: 2, reward: 9998.75
GOAL 1 reached!!! Goals visited: 3, reward: 9999.0
GOAL 3 reached!!! Goals visited: 4, reward: 9999.285714285714
All goals reached!!! Goals visited: [True, True, True, True], reward: 109999.28571428571
Episode ended. total timesteps:43, total rewards: 139904.21428571426
Current observation: [ 3.40875608e+02  1.06265009e+02  2.08924681e+00  5.53865665e+01
  3.98858943e-01  6.18454578e+01  2.83273756e+02 -2.00999515e+00
 -7.02695820e+01 -1.48206971e-01  3.31654778e+02  2.53322778e+02
 -3.18966628e+00 -1.00000000e+02 -2.32083776e-01  1.50000000e+01]


In [9]:
env.close()

Screen record

In [10]:
from pygame_screen_record import ScreenRecorder

env_params = read_env_config(env_config_path)

env = MultiRobotEnv(env_params=env_params, render_mode="human", scale=1.0)
obs, info = env.reset()
print("observation:", obs)
env.render()

recorder = ScreenRecorder(60) # pass your desired fps
recorder.start_rec() # start recording

# Start playing
terminated, truncated = False, False
total_rewards = 0
total_steps = 0
while True:
    action, _ = model.predict(obs) # model prediction
    # action = env.action_space.sample() # random action
    print(action)
    # print(int(action))
    obs, reward, terminated, truncated,  info = env.step(action)
    env.render()
    total_rewards += reward
    print(f"Obs: {obs}, Reward: {reward}, terminated: {terminated}, total_rewards: {total_rewards}, total_steps: {total_steps}")
    if terminated or truncated:
        print('terminated:', terminated, 'truncated:', truncated)
        break
    pygame.event.get()
    total_steps += 1

recorder.stop_rec()	# stop recording
recorder.save_recording("wheeled_robot_trained_inference.mp4") # saves the last recording
pygame.quit()

observation: [ 3.50000000e+02  7.50000000e+01  7.85398163e-01  0.00000000e+00
  0.00000000e+00  1.00000000e+02  1.50000000e+02 -2.61799388e-01
  0.00000000e+00  0.00000000e+00  2.00000000e+02  2.25000000e+02
 -3.12413936e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
[[-0.99658465 -0.44407761]
 [-0.98669946  0.82896495]
 [-0.99667716 -0.98409802]]
Obs: [ 3.49824874e+02  7.48227860e+01  7.91324995e-01 -4.98292327e+00
 -4.44077611e-01  9.97625359e+01  1.50066778e+02 -2.74133131e-01
 -4.93349731e+00  7.85398163e-01  2.00249058e+02  2.25007452e+02
 -3.11168090e+00 -4.98338580e+00 -7.85398163e-01  0.00000000e+00], Reward: -10, terminated: False, total_rewards: -10, total_steps: 0
[[-0.85971594 -0.92773974]
 [-0.99313813 -0.74115807]
 [-0.99970758 -0.93093395]]
Obs: [ 3.49506421e+02  7.44852167e+01  8.14528753e-01 -9.28150296e+00
 -7.85398163e-01  9.92862053e+01  1.50201292e+02 -2.75228698e-01
 -9.89918798e+00  4.42400952e-02  2.00747403e+02  2.25034822e+02
 -3.08672609e+00 -9.98192370